In [1]:
import pandas as pd

data_df = pd.read_csv('asoptimizer_adapted.csv')

In [2]:
data_df.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'ISIS', 'Transfection', 'Linkage',
       'Linkage_Location', 'Canonical Gene Name', 'Cell_line',
       'ASO_volume(nM)', 'Treatment_Period(hours)', 'Sequence', 'Modification',
       'Chemical_Pattern', 'Inhibition(%)', 'probe_std', 'probe_count',
       'total_replicate_count', 'Cohort', 'index_v2', 'Cell line organism',
       'split', 'inhibition_percent', 'aso_sequence_5_to_3', 'dosage',
       'sugar_mods', 'backbone_mods', 'sense_start', 'rna_context',
       'custom_id', 'transfection_method'],
      dtype='object')

In [3]:
from notebooks.consts import OLIGO_CSV_PROCESSED_AVERAGED

oligo_df = pd.read_csv(OLIGO_CSV_PROCESSED_AVERAGED)

In [4]:
from tauso.data.consts import CELL_LINE, CANONICAL_GENE
import pandas as pd

# 1. Isolate the unique combinations from both dataframes
data_unique_pairs = data_df[[CELL_LINE, CANONICAL_GENE]].drop_duplicates()
oligo_unique_pairs = oligo_df[[CELL_LINE, CANONICAL_GENE]].drop_duplicates()

# 2. Merge them to see where they overlap and where they don't
merged_pairs = pd.merge(
    data_unique_pairs,
    oligo_unique_pairs,
    on=[CELL_LINE, CANONICAL_GENE],
    how='outer',
    indicator=True
)

# 3. Filter for combinations that only exist in the left dataframe (ASOPtimizer)
missing_combinations = merged_pairs[merged_pairs['_merge'] == 'left_only'].drop(columns=['_merge'])

print(f"Found {len(missing_combinations)} unique Cell_line x Gene combinations only in ASOPtimizer:")
print(missing_combinations)

Found 9 unique Cell_line x Gene combinations only in ASOPtimizer:
               Cell_line Canonical Gene Name
1                  A-431                IRF5
21                  A431                SOD1
95         Angptl2/Actin             ANGPTL2
97                  H929                IRF4
158  Human Neuronal Cell              SNHG14
164           KARPAS-229                IRF5
165                KMS11                IRF4
172             NCI-H460                KRAS
273                 U251               HTRA1


In [5]:
from tauso.data.consts import CELL_LINE, CANONICAL_GENE

# 1. Count samples for each combination in ASOPtimizer
data_counts = data_df.groupby([CELL_LINE, CANONICAL_GENE]).size().reset_index(name='sample_count')

# 2. Get unique combinations present in the oligo dataframe
oligo_pairs = oligo_df[[CELL_LINE, CANONICAL_GENE]].drop_duplicates()

# 3. Perform a left merge with an indicator to find what is missing in oligo_df
missing_combos = pd.merge(
    data_counts,
    oligo_pairs,
    on=[CELL_LINE, CANONICAL_GENE],
    how='left',
    indicator=True
)

# 4. Filter for rows that only exist in the ASOPtimizer data ('left_only')
missing_combos = missing_combos[missing_combos['_merge'] == 'left_only'].drop(columns=['_merge'])

# 5. Sort by sample count to see the most significant gaps first
missing_combos = missing_combos.sort_values(by='sample_count', ascending=False)

print(f"Found {len(missing_combos)} combinations in ASOPtimizer missing from the oligo dataset:")
print(missing_combos)

Found 9 combinations in ASOPtimizer missing from the oligo dataset:
              Cell_line Canonical Gene Name  sample_count
11  Human Neuronal Cell              SNHG14          1762
5         Angptl2/Actin             ANGPTL2           368
12           KARPAS-229                IRF5           291
18                 U251               HTRA1           126
0                 A-431                IRF5            36
6                  H929                IRF4            17
13                KMS11                IRF4            13
15             NCI-H460                KRAS            12
4                  A431                SOD1             4
